## 1. Parâmetros de exportação

In [0]:
import time
from datetime import datetime

dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("data_referencia_calculo", "2026-05-22")

catalogo = dbutils.widgets.get("catalogo").strip()
data_referencia_calculo = dbutils.widgets.get("data_referencia_calculo").strip()

spark.sql(f"USE CATALOG `{catalogo}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalogo}`.`gold`")

execucao_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Catálogo em uso: {catalogo}")
print(f"Data de referência: {data_referencia_calculo}")
print(f"Execução: {execucao_id}")

## 2. Tabelas Gold exportadas

In [0]:
GOLD_TABLES = [
    "gold_pedidos_enriquecidos",
    "gold_vendas_kpis",
    "gold_produto_performance",
    "gold_cliente_360",
    "gold_tickets",
    "gold_avaliacoes",
    "gold_clickstream_resumo",
]

volume_exports = f"/Volumes/{catalogo}/gold/exports"

spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalogo}`.`gold`.`exports`")

print("Destino dos arquivos:")
print(volume_exports)

print("\nTabelas configuradas:")
for tabela in GOLD_TABLES:
    print(f"- {catalogo}.gold.{tabela}")

## 3. Função de exportação

In [0]:
def exportar_gold_csv(tabela_full: str, volume_path: str):
    inicio = time.time()
    
    nome_tabela = tabela_full.split(".")[-1]
    arquivo_final = f"{volume_path}/{nome_tabela}.csv"
    pasta_temp = f"{volume_path}/_tmp_{nome_tabela}"
    
    if not spark.catalog.tableExists(tabela_full):
        raise Exception(f"Tabela não encontrada: {tabela_full}")
    
    df = spark.table(tabela_full)
    qtd = df.count()
    
    if qtd == 0:
        raise Exception(f"Tabela sem registros: {tabela_full}")
    
    print(f"\nExportando {tabela_full}...")
    
    try:
        dbutils.fs.rm(pasta_temp, recurse=True)
    except:
        pass
    
    (
        df.coalesce(1).write
        .mode("overwrite")
        .option("header", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("nullValue", "")
        .option("emptyValue", "")
        .option("encoding", "UTF-8")
        .option("dateFormat", "yyyy-MM-dd")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .csv(pasta_temp)
    )
    
    arquivos = dbutils.fs.ls(pasta_temp)
    part_file = [
        f.path for f in arquivos
        if f.name.startswith("part-") and f.name.endswith(".csv")
    ][0]
    
    try:
        dbutils.fs.rm(arquivo_final)
    except:
        pass
    
    dbutils.fs.cp(part_file, arquivo_final)
    dbutils.fs.rm(pasta_temp, recurse=True)
    
    duracao = time.time() - inicio
    
    print(f"[OK] {nome_tabela}.csv | linhas={qtd:,} | duração={duracao:.1f}s")
    
    return {
        "tabela": nome_tabela,
        "linhas": qtd,
        "arquivo": arquivo_final,
        "duracao_segundos": round(duracao, 2),
    }

## 4. Execução da exportação

In [0]:
resultados = []

for nome_tabela in GOLD_TABLES:
    tabela_full = f"{catalogo}.gold.{nome_tabela}"
    resultado = exportar_gold_csv(tabela_full, volume_exports)
    resultados.append(resultado)

print(f"\nExports disponíveis em: {volume_exports}")

## 5. Evidência da exportação


In [0]:
df_resumo_export = spark.createDataFrame(resultados)

display(
    df_resumo_export
    .select("tabela", "linhas", "arquivo", "duracao_segundos")
    .orderBy("tabela")
)

print(f"Total de tabelas exportadas: {len(resultados)}")
print(f"Total de linhas exportadas: {sum(r['linhas'] for r in resultados):,}")
print(f"Destino final: {volume_exports}")